# Exploración y Modelado
###### Sistema Actuarial de Precicción de Riesgo y Costo Esperado en seguros de automovil (SAPRICO) 
---

Se sugiere el uso de pipeline y column transformer

In [95]:
import numpy as np
import pandas as pd

from ipywidgets import interact, Dropdown, IntSlider, Checkbox
import altair as alt
import plotly.express as px

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

### 
from sklearn.ensemble import RandomForestRegressor 


### Carga del conjunto de datos

La base de datos sintetica es: `seguro_auto_actuarial.csv`

In [4]:
data = pd.read_csv('../data/seguro_auto_actuarial.csv')

La base consta de las siguietenes  variables

1. *Númericas*

`edad_conductor, ingreso_mensual_mxn,
prima_mensual_mxn, km_anuales,
edad_vehiculo_anios, score_crediticio`


2. *Categóricas nóminales* 

`sexo, estado_civil, ocupacion,
zona_residencia, region, tipo_vehiculo,
uso_vehiculo, metodo_pago`


3. *Categóricas ordinales*

`nivel_estudios, segmento_marca,
deducible_pct ` si se trata como nivel


4. *Binarias*

`tiene_gps, asistencia_vial,
mantenimiento_al_dia, riesgo_alto`


5. *Objetivos* (Target)

`costo_esperado_anual_mxn  (Variable Contiua)` 
Representa el costo anual esperado de siniestros para una póliza.

`riesgo_alto               (Variable Discreta)`
Vale 1 si la póliza pertenece al grupo de mayor riesgo y 0 en caso
contrario. La clase positiva representa aproximadamente 15%, por lo que existe desbalance. 


In [52]:
# Aqui si hay que aprovechar que las variables ya están categorizadas desde la documentación 
# Esto debido a que si se quiere hacer por ciclo, no diferencia adecuadamente 
# Las nominales ni binarias de las que son enteras, y se deberá hacer una exploración más a fondo


numerico = ['edad_conductor', 'ingreso_mensual_mxn',
'prima_mensual_mxn', 'km_anuales','edad_vehiculo_anios'
, 'score_crediticio', 'costo_esperado_anual_mxn']

nominal = ['sexo', 'estado_civil', 'ocupacion',
'zona_residencia', 'region', 'tipo_vehiculo',
'uso_vehiculo', 'metodo_pago']

ordinal = ['nivel_estudios', 'segmento_marca',
'deducible_pct']

binaria = ['tiene_gps', 'asistencia_vial',
'mantenimiento_al_dia', 'riesgo_alto']

categorica = nominal + ordinal + binaria




La base consta de las siguietenes  variables

1. *Númericas*

`edad_conductor, ingreso_mensual_mxn,
prima_mensual_mxn, km_anuales,
edad_vehiculo_anios, score_crediticio`


2. *Categóricas nóminales* 

`sexo, estado_civil, ocupacion,
zona_residencia, region, tipo_vehiculo,
uso_vehiculo, metodo_pago`


3. *Categóricas ordinales*

`nivel_estudios, segmento_marca,
deducible_pct ` si se trata como nivel


4. *Binarias*

`tiene_gps, asistencia_vial,
mantenimiento_al_dia, riesgo_alto`


5. *Objetivos* (Target)

`costo_esperado_anual_mxn  (Variable Contiua)` 
Representa el costo anual esperado de siniestros para una póliza.

`riesgo_alto               (Variable Discreta)`
Vale 1 si la póliza pertenece al grupo de mayor riesgo y 0 en caso
contrario. La clase positiva representa aproximadamente 15%, por lo que existe desbalance. 


La base consta de las siguietenes  variables

1. *Númericas*

`edad_conductor, ingreso_mensual_mxn,
prima_mensual_mxn, km_anuales,
edad_vehiculo_anios, score_crediticio`


2. *Categóricas nóminales* 

`sexo, estado_civil, ocupacion,
zona_residencia, region, tipo_vehiculo,
uso_vehiculo, metodo_pago`


3. *Categóricas ordinales*

`nivel_estudios, segmento_marca,
deducible_pct ` si se trata como nivel


4. *Binarias*

`tiene_gps, asistencia_vial,
mantenimiento_al_dia, riesgo_alto`


5. *Objetivos* (Target)

`costo_esperado_anual_mxn  (Variable Contiua)` 
Representa el costo anual esperado de siniestros para una póliza.

`riesgo_alto               (Variable Discreta)`
Vale 1 si la póliza pertenece al grupo de mayor riesgo y 0 en caso
contrario. La clase positiva representa aproximadamente 15%, por lo que existe desbalance. 


Comprensión del problema 

A partir de las caracteristicas de una cartera de pólizas, poder estimar el costo deribado de su siniestro en pesos mexicanos y clasificar si esta es de alto reisgo. 

Usualmente, el costo del siniestro se utiliza para la determinación del factor de siniestralidad ulitma, que es un componente esencial para el calculo de la reserva, que es la masa monetaria con la que se hara frente a las obligaciones como aseguradora. Toda la metodología de la Reserva de Riesgos en Curso(RRC) y la Incurred But Not Reported(IBNR) esta legislada por la Ley de Instituciones de Seguros y Fianzas (LISF) y métodología documentada en la Circular Unica de Seguros y Fianzas (CUSF)


Seleccionar adecuadamente un riesgo es tambien parte esencial de este trabajo, ya que al no hacerlo estas poneindo en riesgo la integridad de la reserva, es decir, la capaciadad de hacer frente a las obligaciones. Es mejor no prometer nada desde un inicio, que prometer algo que, por limitaciones técnicas ( asuencia de suficiente dinero ) no vas a poder cumplir. Por lo que determinar si un grupo o sujeto no es apto para ser asegurado, porque en sí, ya es casi seguro de que se va a siniestrar es esencial para cumplir promesas, o bien, para poder aplicarles una sobreprima, debido a que no son parte de la masa comun de riesgos.



In [5]:
print('Dimensiones del dataset: ', data.shape)
##  Tipos de datos y nulos 
print(data.info())

Dimensiones del dataset:  (1500, 31)
<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   poliza_id                     1500 non-null   str    
 1   edad_conductor                1500 non-null   int64  
 2   sexo                          1500 non-null   str    
 3   estado_civil                  1500 non-null   str    
 4   nivel_estudios                1500 non-null   str    
 5   ocupacion                     1419 non-null   str    
 6   zona_residencia               1500 non-null   str    
 7   region                        1500 non-null   str    
 8   antiguedad_cliente_anios      1500 non-null   float64
 9   ingreso_mensual_mxn           1394 non-null   float64
 10  score_crediticio              1415 non-null   float64
 11  prima_mensual_mxn             1500 non-null   float64
 12  suma_asegurada_mxn            1500 n

In [33]:
# 

null_por_col = data.isnull().sum() / len(data) 

print('Proporción de núlos por columna\n','\n')
null_por_col[null_por_col>0].sort_values(ascending= False).round(4)*100

Proporción de núlos por columna
 



ingreso_mensual_mxn     7.07
score_crediticio        5.67
ocupacion               5.40
puntaje_riesgo_zona     3.53
mantenimiento_al_dia    3.07
segmento_marca          2.93
dtype: float64

In [93]:
## Se crea una funcion para podre pasarla a utils

# nbins es el número de intervalos que va a hacer en el continuo de los valores numericos

def histograma_plotly(variable_num, variable_cat, nbins = 40, por_cateogria = True):
    # Generamos el histograma con el marginal tipo box, pues esto hace posible ver los outliers mediante el método de IQR
    
    # Si queremos diferenciarlos por cateogrías
    if por_cateogria:
        fig = px.histogram(
            data,           
            x=variable_num, 
            color=variable_cat, # Diferenciar los datos por color mediante su categoría
            marginal="box",     # Agrega boxplot, o diagrama de caja y bigotes para ver el IQR  y por ende los outliers         
            barmode="overlay",  # Hace que se hagan menos opacos los histogrammas para verlos mejor         
            nbins= nbins,                     
            title=f"Distribución de {variable_num.replace('_', ' ').title()} por {variable_cat.replace('_', ' ').title()}",
            template="plotly_white",      # Un fondo blanco limpio y estético
            color_discrete_sequence=px.colors.qualitative.Safe # Paleta de colores moderna
        )

    # O bien, ver la distribución de la muestra para cada categoría numerica
    else: 
         fig = px.histogram(
            data, 
            x=variable_num, 
         
            marginal="box",               
            barmode="overlay",            
            nbins= nbins,                     
            title=f"Distribución de {variable_num.replace('_', ' ').title()}",
            template="plotly_white",      # Un fondo blanco limpio y estético
            color_discrete_sequence=px.colors.qualitative.Safe # Paleta de colores moderna       
        )
         
    fig.update_xaxes(title_text = variable_num.replace('_', ' ').title())
    fig.update_yaxes(title_text = 'Recuento')

    fig.update_layout(
    legend_title_text = variable_cat.replace('_', ' ').title(), 
    height=1000,
    title_font_size=18,
    hovermode="x unified"  # Agrupa el tooltip para ver todas las categorías juntas al pasar el mouse
    )
    
    fig.show()

In [105]:
## Exploración de los datos previamente mediante histogramas

interact(
    histograma_plotly,
    variable_num = Dropdown(options = numerico,value = 'costo_esperado_anual_mxn', description = 'Numérica'),
    variable_cat = Dropdown(options = categorica, value = 'zona_residencia', description = 'Categoría:'),
    nbins = IntSlider(min = 10, max = 250, step = 1, value = 40, description = 'Intervalos: '),
    por_cateogria = Checkbox(description = 'Categorias', value = True)
)

interactive(children=(Dropdown(description='Numérica', index=6, options=('edad_conductor', 'ingreso_mensual_mx…

<function __main__.histograma_plotly(variable_num, variable_cat, nbins=40, por_cateogria=True)>